In [5]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean
from plant_gravitropism import generate_smart_grid, get_top_n_with_ties, compute_step

In [6]:
# -------------------------
# Helper to build toy dataframes
# -------------------------
def make_df(rows):
    return pd.DataFrame([{
        'arbor type': 'optimal',
        'G': g,
        'alpha': a,
        'total orthogonal distance': err,
        'total squared orthogonal distance': err**2
    } for g, a, err in rows])

In [7]:
# -------------------------
# Test 1: Ties at rank boundary
# 5 tied for 1st, 5 for 2nd, 5 for 3rd
# smart_num=3 should refine around all 15
# -------------------------
print("=" * 60)
print("TEST 1: Ties at rank boundary (smart_num=3)")
print("Expected: 15 refinement centers")
print("=" * 60)

rows1 = (
    [(-1.0 + i*0.2, 0.3, 0.1) for i in range(5)] +  # 5 tied 1st
    [(-1.0 + i*0.2, 0.5, 0.2) for i in range(5)] +  # 5 tied 2nd
    [(-1.0 + i*0.2, 0.7, 0.3) for i in range(5)] +  # 5 tied 3rd
    [(0.0, 0.5, 1.0), (1.0, 0.5, 2.0)]               # worse points
)
df1 = make_df(rows1)
params1, skip1 = generate_smart_grid(df1, smart_num=3, grid_size=3)
centers_used = len([r for _, r in df1[df1['arbor type']=='optimal'].iterrows()
                    if r['total orthogonal distance'] <= 0.3])
print(f"\nRefinement centers used: {centers_used} (expected 15)")
print(f"PASS: {centers_used == 15}\n")


TEST 1: Ties at rank boundary (smart_num=3)
Expected: 15 refinement centers
Refining around (-1.0, 0.3) | dist=0.2000, step=0.0418 [G_min boundary — extending below]
Refining around (-0.8, 0.3) | dist=0.2000, step=0.0418
Refining around (-0.6, 0.3) | dist=0.2000, step=0.0418
Refining around (-0.3999999999999999, 0.3) | dist=0.2000, step=0.0418
Refining around (-0.19999999999999996, 0.3) | dist=0.2000, step=0.0418

Refinement centers used: 15 (expected 15)
PASS: True



In [8]:
# -------------------------
# Test 2: Best point at G_max boundary
# Best is at G=2.0, should generate G > 2.0
# -------------------------
print("=" * 60)
print("TEST 2: Best point at G_max boundary")
print("Expected: some generated G > 2.0")
print("=" * 60)

rows2 = [(-2.0, 0.5, 2.0), (-1.0, 0.5, 1.5),
         (0.0, 0.5, 1.0), (1.0, 0.5, 0.5), (2.0, 0.5, 0.1)]
df2 = make_df(rows2)
params2, skip2 = generate_smart_grid(df2, smart_num=1, grid_size=3)
G_vals2 = [g for g, a in params2]
print(f"\nMax G generated: {max(G_vals2):.4f} (expected > 2.0)")
print(f"PASS: {max(G_vals2) > 2.0}\n")

TEST 2: Best point at G_max boundary
Expected: some generated G > 2.0
Refining around (2.0, 0.5) | dist=1.0000, step=0.1000 [G_max boundary — extending above]

Max G generated: 2.1000 (expected > 2.0)
PASS: True



In [9]:
# -------------------------
# Test 3: Best point at G_min boundary
# Best is at G=-2.0, should generate G < -2.0
# -------------------------
print("=" * 60)
print("TEST 3: Best point at G_min boundary")
print("Expected: some generated G < -2.0")
print("=" * 60)

rows3 = [(-2.0, 0.5, 0.1), (-1.0, 0.5, 0.5),
         (0.0, 0.5, 1.0), (1.0, 0.5, 1.5), (2.0, 0.5, 2.0)]
df3 = make_df(rows3)
params3, skip3 = generate_smart_grid(df3, smart_num=1, grid_size=3)
G_vals3 = [g for g, a in params3]
print(f"\nMin G generated: {min(G_vals3):.4f} (expected < -2.0)")
print(f"PASS: {min(G_vals3) < -2.0}\n")

TEST 3: Best point at G_min boundary
Expected: some generated G < -2.0
Refining around (-2.0, 0.5) | dist=1.0000, step=0.1000 [G_min boundary — extending below]

Min G generated: -2.1000 (expected < -2.0)
PASS: True



In [10]:
# -------------------------
# Test 4: Best point at alpha=0 boundary
# Alpha must not go below 0
# -------------------------
print("=" * 60)
print("TEST 4: Best point at alpha=0 boundary")
print("Expected: all generated alpha >= 0")
print("=" * 60)

rows4 = [(0.0, 0.0, 0.1), (0.0, 0.25, 0.5),
         (0.0, 0.5, 1.0), (0.0, 0.75, 1.5), (0.0, 1.0, 2.0)]
df4 = make_df(rows4)
params4, skip4 = generate_smart_grid(df4, smart_num=1, grid_size=3)
alpha_vals4 = [a for g, a in params4]
print(f"\nMin alpha generated: {min(alpha_vals4):.4f} (expected >= 0.0)")
print(f"PASS: {min(alpha_vals4) >= 0.0}\n")

TEST 4: Best point at alpha=0 boundary
Expected: all generated alpha >= 0
Refining around (0.0, 0.0) | dist=0.2500, step=0.0515 [G_min boundary — extending below, G_max boundary — extending above, alpha=0 boundary — clamped]

Min alpha generated: 0.0000 (expected >= 0.0)
PASS: True



In [11]:
# -------------------------
# Test 5: Best point at alpha=1 boundary
# Alpha must not go above 1
# -------------------------
print("=" * 60)
print("TEST 5: Best point at alpha=1 boundary")
print("Expected: all generated alpha <= 1")
print("=" * 60)

rows5 = [(0.0, 1.0, 0.1), (0.0, 0.75, 0.5),
         (0.0, 0.5, 1.0), (0.0, 0.25, 1.5), (0.0, 0.0, 2.0)]
df5 = make_df(rows5)
params5, skip5 = generate_smart_grid(df5, smart_num=1, grid_size=3)
alpha_vals5 = [a for g, a in params5]
print(f"\nMax alpha generated: {max(alpha_vals5):.4f} (expected <= 1.0)")
print(f"PASS: {max(alpha_vals5) <= 1.0}\n")

print("=" * 60)
print("All tests complete")
print("=" * 60)

TEST 5: Best point at alpha=1 boundary
Expected: all generated alpha <= 1
Refining around (0.0, 1.0) | dist=0.2500, step=0.0515 [G_min boundary — extending below, G_max boundary — extending above, alpha=1 boundary — clamped]

Max alpha generated: 1.0000 (expected <= 1.0)
PASS: True

All tests complete
